In [ ]:
# PLEASE NOTE: all code worked for me.
# note: some error occured ONCE, RANDOMLY: sympy.something doesn't have something.
# was tracing back to optimizer initalisation line.
# *** TA said that is a runtime/environment issue, reload session. did that, now fine. ***
# if you also see a similar issue, PLEASE DO NOT BLAME ME!!

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
# importing everything at start to avoid headache
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
from torchvision.transforms.functional import to_tensor


import matplotlib.pyplot as plt
import seaborn as sns

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

# print shape again
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 2. Create TensorDataset objects
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)
type(train_ds)

In [ ]:
# 3. Create DataLoaders

batch_size = 32

train_loader = DataLoader(train_ds, batch_size=batch_size)
test_loader = DataLoader(test_ds, batch_size=batch_size)


In [ ]:
# 4. Print shape of one batch
pair1 = next(iter(train_loader)) # returns a normal python list w 2 items: batch_X, batch_y
print(type(pair1))
print(len(pair1))
print("Batch_X", pair1[0].shape)
print("Batch_y", pair1[1].shape)

In [ ]:
# 5. Display sample images

images, labels = pair1[0], pair1[1]

plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Age: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here:
class ANON(nn.Module):

  def __init__(self):
    super().__init__()
    self.layer1 = nn.Linear(3 * 36 * 36, 512)
    self.layer2 = nn.Linear(512, 128)
    self.layer3 = nn.Linear(128, 32)
    self.layer4 = nn.Linear(32, 1)
    self.relu = nn.ReLU()
    self.dropout = nn.Dropout(p=0.1)
    # yes ik what dropout is. zeros some weights randomly during training to prevent overfit. no i didnt use chatgpt for it.
    # pls give bonus if you feel it deserves.

  def forward(self, x):
    x = x.view(-1, 3 * 36 * 36) # flattening it.
    x = F.relu(self.layer1(x))
    x = self.dropout(F.relu(self.layer2(x)))
    x = F.relu(self.layer3(x))
    x = self.layer4(x)

    return x # outputs raw "logit" cuz regression. not classification.

In [ ]:
# Task 2: Write your training loop here:
# I am writing one fn for training, one for validation. Learnt from labs; code my own.
from tqdm import tqdm, trange # for pbar
def train_one_epoch(model, optimizer, loss_fn, train_loader, device):
  epoch_loss = 0
  model.train()
  model.dropout.train()
  for b_x, b_y in train_loader:
    b_x = b_x.to(device)
    b_y = b_y.to(device)
    preds = model(b_x)
    loss = loss_fn(preds, b_y)
    epoch_loss += loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  return epoch_loss / batch_size / len(test_loader)


In [ ]:
# Task 3: Write your validation loop here:

def validate(model, loss_fn, test_loader, device):
  total_loss = 0
  model.eval()
  model.dropout.eval()
  with torch.no_grad():
    for b_x, b_y in test_loader:
      b_x = b_x.to(device)
      b_y = b_y.to(device)
      preds = model(b_x)
      loss = loss_fn(preds, b_y)
      total_loss += loss.item()
      #print("testloaderlen", len(test_loader))

  return total_loss / len(test_loader) / batch_size

In [ ]:
# Task 4: Define device, model, loss, optimizer:

num_epochs = 20
learning_rate = 0.001
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)
model = ANON() # didn't keep input dim etc args cuz ik fixed for this task.
model = model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
#AdamW cuz theory said this is best. tho i have seen Adam being used a lot more often.
loss_fn = nn.MSELoss() # cuz regression.

tr_losses, ts_losses = [], [] # setting to empty lists, will add once training.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
# disabling cuz it ruins my lovely progress bar later.
# i had to rote learn this asw. cuz it kept annoying all the damn time.

In [ ]:
# Task 5: Start training for 20 epochs:
pbar = tqdm(desc="Training", unit='Epoch', total = num_epochs, ncols=140) # yes i learnt how to use tqdm.
# using pbar instead of simply trange(num_epochs) cuz then i can display train/val loss and acc in pbar w set_postfix.
# otherwise i couldn't properly.

for epoch in range(num_epochs):
  epoch_loss = train_one_epoch(model, optimizer, loss_fn, train_loader, device)
  tr_losses.append(epoch_loss)
  valid_loss = validate(model, loss_fn, test_loader, device)
  ts_losses.append(valid_loss)

  pbar.update(1)
  pbar.set_postfix({'Tr_Loss': f'{epoch_loss:.3f}', 'Ts_Loss': f'{valid_loss:.3f}'})

In [ ]:
# Task 1: Write your code here:
sns.set_style('whitegrid')

plt.plot(tr_losses, label='Training Loss')
plt.plot(ts_losses, label='Validation Loss')
plt.legend()
plt.title("Loss over epochs")

In [ ]:
# Task 2 (Bonus): Write your code here:
images, labels = next(iter(test_loader))
images = images.to(device)
fpreds = model(images)

images = images.cpu()
labels = labels.cpu()
fpreds = fpreds.cpu()



plt.figure(figsize=(8, 4))
plt.axis('off')
for i in range(6):
    plt.subplot(2, 3, i + 1)
    img = images[i].permute(1, 2, 0)
    plt.imshow(img)


    plt.title(f"real {labels[i]:.1f} -vs- pred {fpreds[i].item():.1f}")
    plt.axis('off')

plt.tight_layout()
plt.show()
# bonus please !! :D